# Notebook 05 — Bonus: Database Sink (SQLite)

**Goal:** load the nine aggregate CSV files into `data/warehouse.db` for use in Power BI.

This notebook uses the explicit schema in `sql/schema.sql`, transaction-safe table loads, an audit trail, data verification, an idempotency check, and a rollback test.

Run the notebook from top to bottom. Do not run cells out of order.

In [1]:
import sqlite3
from datetime import datetime
from pathlib import Path

import pandas as pd

CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
DB_PATH = PROJECT_ROOT / "data" / "warehouse.db"
SCHEMA_PATH = PROJECT_ROOT / "sql" / "schema.sql"

TABLES = [
    "agg_ridership_by_month",
    "agg_ridership_by_hour",
    "agg_route_performance",
    "agg_mode_summary",
    "agg_fare_type_mix",
    "agg_stop_activity",
    "agg_vehicle_utilisation",
    "agg_peak_analysis",
    "agg_transfer_behaviour",
]

assert PROCESSED_DIR.is_dir(), f"Processed data folder not found: {PROCESSED_DIR}"
assert SCHEMA_PATH.is_file(), f"SQL schema file not found: {SCHEMA_PATH}"

for table in TABLES:
    csv_path = PROCESSED_DIR / f"{table}.csv"
    assert csv_path.is_file(), f"Required CSV file not found: {csv_path}"

print("Project root:", PROJECT_ROOT)
print("Database path:", DB_PATH)
print("Processed data folder:", PROCESSED_DIR)

Project root: c:\Users\User\OneDrive\Desktop\kurs\CityTransit_capstone_FINAL_corrected
Database path: c:\Users\User\OneDrive\Desktop\kurs\CityTransit_capstone_FINAL_corrected\data\warehouse.db
Processed data folder: c:\Users\User\OneDrive\Desktop\kurs\CityTransit_capstone_FINAL_corrected\data\processed


## 1. Create the database schema

The schema is defined in `sql/schema.sql`. The connection uses a timeout to wait briefly if SQLite is busy. Only one connection is opened in this notebook.

In [2]:
def connect() -> sqlite3.Connection:
    connection = sqlite3.connect(
        DB_PATH,
        timeout=30,
        isolation_level=None
    )
    connection.execute("PRAGMA foreign_keys = ON")
    return connection


conn = connect()
conn.executescript(SCHEMA_PATH.read_text(encoding="utf-8"))

schema_objects = pd.read_sql_query(
    """
    SELECT type, name
    FROM sqlite_master
    WHERE name NOT LIKE 'sqlite_%'
    ORDER BY type DESC, name
    """,
    conn
)

display(schema_objects)

table_count = (schema_objects["type"] == "table").sum()
index_count = (schema_objects["type"] == "index").sum()

print("Database tables:", table_count)
print("Database indexes:", index_count)

,type,name
0,table,agg_fare_type_mix
1,table,agg_mode_summary
2,table,agg_peak_analysis
3,table,agg_ridership_by_hour
4,table,agg_ridership_by_month
5,table,agg_route_performance
6,table,agg_stop_activity
7,table,agg_transfer_behaviour
8,table,agg_vehicle_utilisation
9,table,pipeline_run_tables


Database tables: 11
Database indexes: 8


## 2. Define the loading functions

The monthly aggregate CSV contains five columns. The database schema also requires `month_start` and `boardings_mom_growth`; these derived columns are added during loading. Every other aggregate CSV already carries every column the schema needs (notebook 04 adds them at export time), so no further derivation is required.

Each table is replaced inside its own transaction. If an insert fails, SQLite rolls back the deletion and preserves the previous table contents.

In [3]:
def table_columns(connection: sqlite3.Connection, table: str) -> list[str]:
    return [
        row[1]
        for row in connection.execute(f"PRAGMA table_info({table})")
    ]


def prepare_table_data(table: str, dataframe: pd.DataFrame) -> pd.DataFrame:
    prepared = dataframe.copy()

    if table == "agg_ridership_by_month":
        if "month_start" not in prepared.columns:
            prepared["month_start"] = pd.to_datetime(
                prepared["year_month"] + "-01"
            ).dt.strftime("%Y-%m-%d")

        if "boardings_mom_growth" not in prepared.columns:
            prepared["boardings_mom_growth"] = (
                prepared["total_boardings"].pct_change()
            )

    database_columns = table_columns(conn, table)
    missing_columns = set(database_columns) - set(prepared.columns)

    if missing_columns:
        raise ValueError(
            f"{table}: missing required database columns: "
            f"{sorted(missing_columns)}"
        )

    return prepared[database_columns]


def load_table(
    connection: sqlite3.Connection,
    table: str,
    dataframe: pd.DataFrame
) -> int:
    database_columns = table_columns(connection, table)
    prepared = prepare_table_data(table, dataframe)

    rows = (
        prepared.astype(object)
        .where(prepared.notna(), None)
        .values.tolist()
    )

    placeholders = ", ".join("?" for _ in database_columns)
    column_sql = ", ".join(database_columns)
    insert_sql = (
        f"INSERT INTO {table} ({column_sql}) "
        f"VALUES ({placeholders})"
    )

    connection.execute("BEGIN")
    try:
        connection.execute(f"DELETE FROM {table}")
        connection.executemany(insert_sql, rows)
        connection.execute("COMMIT")
    except Exception:
        connection.execute("ROLLBACK")
        raise

    return len(rows)


def run_pipeline(connection: sqlite3.Connection) -> tuple[int, str]:
    started_at = datetime.now().isoformat(timespec="seconds")

    run_id = connection.execute(
        "INSERT INTO pipeline_runs (started_at, status) "
        "VALUES (?, 'RUNNING')",
        (started_at,)
    ).lastrowid

    status = "SUCCESS"
    error_message = None

    try:
        for table in TABLES:
            csv_path = PROCESSED_DIR / f"{table}.csv"
            dataframe = pd.read_csv(csv_path)
            rows_written = load_table(connection, table, dataframe)

            connection.execute(
                """
                INSERT INTO pipeline_run_tables
                    (run_id, table_name, rows_written)
                VALUES (?, ?, ?)
                """,
                (run_id, table, rows_written)
            )

    except Exception as exc:
        status = "FAILED"
        error_message = f"{type(exc).__name__}: {exc}"

    connection.execute(
        """
        UPDATE pipeline_runs
        SET finished_at = ?, status = ?, error_message = ?
        WHERE run_id = ?
        """,
        (
            datetime.now().isoformat(timespec="seconds"),
            status,
            error_message,
            run_id
        )
    )

    print(
        f"Pipeline run {run_id}: {status}"
        + (f" — {error_message}" if error_message else "")
    )

    return run_id, status

## 3. Run the data load

In [4]:
run_id, run_status = run_pipeline(conn)

assert run_status == "SUCCESS", (
    "The database load failed. Check the pipeline status and error message."
)

run_details = pd.read_sql_query(
    """
    SELECT table_name, rows_written
    FROM pipeline_run_tables
    WHERE run_id = ?
    ORDER BY table_name
    """,
    conn,
    params=(run_id,)
)

display(run_details)
assert len(run_details) == len(TABLES), "Not all nine aggregate tables were loaded."

Pipeline run 3: SUCCESS


,table_name,rows_written
0,agg_fare_type_mix,30
1,agg_mode_summary,18
2,agg_peak_analysis,18
3,agg_ridership_by_hour,48
4,agg_ridership_by_month,6
5,agg_route_performance,360
6,agg_stop_activity,400
7,agg_transfer_behaviour,18
8,agg_vehicle_utilisation,180


## 4. Verify the database against the CSV files

For each table, compare the row count and every data column. The monthly CSV is enriched with the same two derived columns used by the loader.

In [5]:
verification_results = []

for table in TABLES:
    csv_dataframe = pd.read_csv(PROCESSED_DIR / f"{table}.csv")
    expected_dataframe = prepare_table_data(table, csv_dataframe)

    database_dataframe = pd.read_sql_query(
        f"SELECT * FROM {table}",
        conn
    )

    row_count_matches = len(expected_dataframe) == len(database_dataframe)
    data_matches = row_count_matches

    if row_count_matches:
        try:
            pd.testing.assert_frame_equal(
                expected_dataframe.reset_index(drop=True),
                database_dataframe[expected_dataframe.columns].reset_index(drop=True),
                check_dtype=False,
                check_exact=False,
                rtol=1e-9,
                atol=1e-9
            )
        except AssertionError:
            data_matches = False

    verification_results.append({
        "table": table,
        "csv_rows": len(expected_dataframe),
        "database_rows": len(database_dataframe),
        "row_count_match": row_count_matches,
        "data_match": data_matches
    })

verification = pd.DataFrame(verification_results)
verification["match"] = (
    verification["row_count_match"] & verification["data_match"]
)

display(verification)

assert verification["match"].all(), (
    "At least one database table does not match its processed CSV."
)

print("All nine aggregate tables match their CSV source data.")

,table,csv_rows,database_rows,row_count_match,data_match,match
0,agg_ridership_by_month,6,6,True,True,True
1,agg_ridership_by_hour,48,48,True,True,True
2,agg_route_performance,360,360,True,True,True
3,agg_mode_summary,18,18,True,True,True
4,agg_fare_type_mix,30,30,True,True,True
5,agg_stop_activity,400,400,True,True,True
6,agg_vehicle_utilisation,180,180,True,True,True
7,agg_peak_analysis,18,18,True,True,True
8,agg_transfer_behaviour,18,18,True,True,True


All nine aggregate tables match their CSV source data.


## 5. Idempotency check

Run the pipeline a second time. Because each aggregate table is replaced transactionally, the row counts must remain unchanged.

In [6]:
expected_row_counts = {
    row["table"]: row["csv_rows"]
    for _, row in verification.iterrows()
}

second_run_id, second_run_status = run_pipeline(conn)
assert second_run_status == "SUCCESS", "The second pipeline run failed."

second_run_counts = {
    table: conn.execute(
        f"SELECT COUNT(*) FROM {table}"
    ).fetchone()[0]
    for table in TABLES
}

assert second_run_counts == expected_row_counts, (
    "The second run changed one or more table row counts."
)

print("IDEMPOTENCY TEST PASSED")
print("All nine table row counts remain unchanged.")

Pipeline run 4: SUCCESS
IDEMPOTENCY TEST PASSED
All nine table row counts remain unchanged.


## 6. Transaction rollback test

Attempt to load a deliberately invalid monthly record with a null `year_month`. The database's `NOT NULL` constraint must reject it, and the transaction must preserve the original monthly records.

In [7]:
monthly_table = "agg_ridership_by_month"

rows_before = conn.execute(
    f"SELECT COUNT(*) FROM {monthly_table}"
).fetchone()[0]

invalid_monthly_data = pd.DataFrame([{
    "year_month": None,
    "total_boardings": 100,
    "total_revenue": 150.0,
    "unique_cards": 80,
    "avg_daily_boardings": 10.0,
    "month_start": "2024-01-01",
    "boardings_mom_growth": None
}])

rollback_error_caught = False

try:
    load_table(conn, monthly_table, invalid_monthly_data)
except sqlite3.IntegrityError as exc:
    rollback_error_caught = True
    print("Invalid load rejected as expected:", exc)

rows_after = conn.execute(
    f"SELECT COUNT(*) FROM {monthly_table}"
).fetchone()[0]

print("Rows before:", rows_before)
print("Rows after rollback:", rows_after)

assert rollback_error_caught, "The invalid record was not rejected."
assert rows_before == rows_after, "Rollback did not preserve the original rows."

print("ROLLBACK TEST PASSED")

Invalid load rejected as expected: NOT NULL constraint failed: agg_ridership_by_month.year_month
Rows before: 6
Rows after rollback: 6
ROLLBACK TEST PASSED


## 7. Audit trail

In [8]:
audit_trail = pd.read_sql_query(
    """
    SELECT
        r.run_id,
        r.started_at,
        r.finished_at,
        r.status,
        COUNT(t.table_name) AS tables_loaded,
        COALESCE(SUM(t.rows_written), 0) AS rows_written,
        r.error_message
    FROM pipeline_runs AS r
    LEFT JOIN pipeline_run_tables AS t
        ON t.run_id = r.run_id
    GROUP BY r.run_id
    ORDER BY r.run_id DESC
    LIMIT 10
    """,
    conn
)

display(audit_trail)

,run_id,started_at,finished_at,status,tables_loaded,rows_written,error_message
0,4,2026-11-22T18:19:35,2026-11-22T18:19:35,SUCCESS,9,1078,None
1,3,2026-11-22T18:19:35,2026-11-22T18:19:35,SUCCESS,9,1078,None
2,2,2026-11-20T14:33:17,2026-11-20T14:33:17,SUCCESS,9,1078,None
3,1,2026-11-20T14:33:17,2026-11-20T14:33:17,SUCCESS,9,1078,None


## 8. Example SQL query for Power BI

In [9]:
power_bi_example = pd.read_sql_query(
    """
    SELECT
        m.year_month,
        m.mode,
        m.total_boardings,
        m.boardings_per_route,
        ROUND(
            m.total_boardings * 1.0 / NULLIF(t.total_boardings, 0),
            4
        ) AS share_of_network,
        tr.transfer_rate
    FROM agg_mode_summary AS m
    JOIN agg_ridership_by_month AS t
        ON t.year_month = m.year_month
    JOIN agg_transfer_behaviour AS tr
        ON tr.year_month = m.year_month
        AND tr.mode = m.mode
    WHERE m.year_month = '2024-06'
    ORDER BY m.total_boardings DESC
    """,
    conn
)

display(power_bi_example)

,year_month,mode,total_boardings,boardings_per_route,share_of_network,transfer_rate
0,2024-06,Bus,7729,227.3235,0.5551,0.3897
1,2024-06,Tram,4788,239.4000,0.3439,0.3763
2,2024-06,Metro,1407,234.5000,0.1010,0.3753


## 9. Connect Power BI

1. Install a SQLite ODBC driver and create a DSN that points to `data/warehouse.db`.
2. In Power BI Desktop, select **Get Data → ODBC**.
3. Choose the DSN and select the required `agg_*` tables.
4. Build or refresh report visuals using the database tables.

Power BI Desktop runs on Windows; the connection and report setup are completed in Power BI, not in this notebook.

In [10]:
conn.close()
print("Database connection closed.")

Database connection closed.
